# Fetch a TTT-E2E checkpoint (and the Books3 `/val` subset) to Google Drive

**Runtime: CPU (free).** Runtime -> Change runtime type -> CPU. No GPU is needed to download.

**Who runs this:** anyone with a billing-enabled Google Cloud project (for session 1: Jaykay,
under his own Google login). **No service-account key is created or shared.** Only the
downloaded files leave Colab, through Drive.

Run the cells **in order**. Cells 2 and 3 are free, metadata-only checks. Each one stops the
notebook with a clear message if it fails, so nothing is billed and nothing downloads.

- Both buckets are **requester-pays**: every request must name a Google Cloud project with
  billing enabled (`gsutil -u <project>`). Cell 2 checks whether being signed in is enough.
- Default target is **125M** (`PREREGISTERED.md` revision 2026-09-22). That's 0.68 GB for the
  checkpoint plus about 0.4 GB for the `/val` chunk, so about ₹15 of egress. Set `CKPT` in
  cell 2 to fetch the 1B instead (5.35 GB).
- Cell 4 fetches the checkpoint and fingerprints it. Cell 5 fetches the `/val` subset. Both
  land as one tar plus a sha256 manifest in `MyDrive/ttt/`.

In [ ]:
# Cell 1 - sign in with your Google account
from google.colab import auth
auth.authenticate_user()
!gcloud auth list --filter=status:ACTIVE --format="value(account)"

In [ ]:
# Cell 2 - signed in, but WITHOUT a billing project: does the bucket let us read?
import subprocess
CKPT = "125m_ttt_e2e_finetune_books_8k_1x_cc"   # or "1b_ttt_e2e_finetune_books_8k_1x_cc"
VAL_TOKENS = 50_000_000                         # TOLERANCE.md 2026-09-16: one 100M-token chunk

# Byte counts measured by scripts/probe_gcs_access.sh (results/gcs-probe-20260914T164647Z.txt)
# and by the 1B download. A mismatch means the wrong path or a changed bucket: stop.
EXPECTED_BYTES = {
    "125m_ttt_e2e_finetune_books_8k_1x_cc": 683046332,
    "1b_ttt_e2e_finetune_books_8k_1x_cc": 5347020507,
}
if CKPT not in EXPECTED_BYTES:
    raise SystemExit(f"STOP: no measured byte count for {CKPT}. Add one from a probe first.")
EXPECTED = EXPECTED_BYTES[CKPT]
SRC = f"gs://ttt-e2e-checkpoints/{CKPT}"
r = subprocess.run(["gsutil", "du", "-s", SRC], capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode == 0:
    print("OK: readable without a billing project. Leave PROJECT empty in cell 3 and continue.")
    NEED_PROJECT = False
elif "requester pays" in (r.stdout + r.stderr).lower() or "UserProjectMissing" in (r.stdout + r.stderr):
    NEED_PROJECT = True
    print("RESULT: signed in is NOT enough - the bucket needs a billing project. Go to cell 3.")
else:
    NEED_PROJECT = True
    print("Unexpected error above (auth?). Read it before continuing.")
print()
print("Projects this account can see (billing must be enabled on the one you use):")
!gcloud projects list --format="table(projectId,name)" 2>&1 | head -20

In [ ]:
# Cell 3 - only needed if cell 2 said a billing project is required
PROJECT = ""   # <- a project ID from the list above, with billing enabled. Leave "" if cell 2 said OK.

import subprocess
ACCESS_OK = False
if NEED_PROJECT and not PROJECT:
    raise SystemExit("STOP: this bucket needs a billing-enabled Google Cloud project, and PROJECT is empty. "
                     "Nothing was downloaded or billed.")
GS = ["gsutil"] + (["-u", PROJECT] if NEED_PROJECT else [])
r = subprocess.run(GS + ["du", "-s", SRC], capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode != 0 or not r.stdout.strip().startswith(str(EXPECTED)):
    raise SystemExit(f"STOP: access check failed (expected {EXPECTED} bytes). Do not run cell 4.")
print(f"ACCESS OK: {EXPECTED} bytes. Cell 4 will download (egress billed to", PROJECT or "nobody", ").")
ACCESS_OK = True

In [ ]:
# Cell 4 - download to /content, fingerprint, copy to Drive. Refuses to run unless cell 3 printed ACCESS OK.
import os, shutil, subprocess, glob
if not globals().get("ACCESS_OK"):
    raise SystemExit("STOP: cell 3 did not pass (ACCESS OK not printed). Nothing to download.")

def run(cmd, **kw):
    # Colab does not show a child process's stdout/stderr, so capture and print it:
    # without this a failing script's own error message never reaches the notebook.
    print("$", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout, end="")
    print(r.stderr, end="")
    if r.returncode != 0:   # any failure stops the cell: no false DONE
        raise SystemExit(f"STOP: exit {r.returncode} from the command above. Its output is printed above.")

from google.colab import drive
drive.mount("/content/drive")
free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"Drive free: {free_gb:.1f} GB")
need_gb = EXPECTED / 1e9 * 1.2 + 1.0   # tar + manifest, plus the /val tar from cell 5
if free_gb < need_gb:
    raise SystemExit(f"STOP: need ~{need_gb:.1f} GB free in Drive. Free space first.")

shutil.rmtree("/content/TTT", ignore_errors=True)
run(["git", "clone", "-q", "https://github.com/Manas-Maahir/Trust-Gated-Fast-Weight-Updates-for-TTT-E2E-LLMs.git", "/content/TTT"])
run(["git", "-C", "/content/TTT", "checkout", "-q", "infra/gpu-session-1"])

DEST = f"/content/ttt-handoff/{CKPT}"
os.makedirs(DEST, exist_ok=True)
run(GS + ["-m", "cp", "-n", "-r", f"{SRC}/*", f"{DEST}/"])

got = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(DEST) for f in fs)
print("downloaded bytes:", got)
if got != EXPECTED:
    raise SystemExit(f"STOP: expected {EXPECTED} bytes, got {got}. Re-run this cell (cp -n resumes).")

# Same sha256 manifest the GPU box verifies against (CKPT_SHA_MANIFEST).
env = dict(os.environ, DEST=DEST, CKPT=CKPT, RESULTS="/content/ttt-handoff", SOURCE=SRC, REMOTE_BYTES=str(EXPECTED))
run(["bash", "/content/TTT/scripts/fingerprint_checkpoint.sh"], env=env)
manifests = glob.glob("/content/ttt-handoff/checkpoint-sha256-*.txt")
if not manifests:
    raise SystemExit("STOP: no checksum manifest was written.")

os.makedirs("/content/drive/MyDrive/ttt", exist_ok=True)
run(["tar", "-cf", f"/content/drive/MyDrive/ttt/ttt-handoff-{CKPT}.tar", "-C", "/content", "ttt-handoff"])
for m in manifests:
    shutil.copy(m, "/content/drive/MyDrive/ttt/")
print(f"DONE: Drive -> MyDrive/ttt/ttt-handoff-{CKPT}.tar and", [os.path.basename(m) for m in manifests])

In [ ]:
# Cell 5 - the Books3 /val subset and /train chunk 0 (same billing project), truncated and manifested by the repo's
# own script, so the laptop's bootstrap accepts it with SKIP_DATA=1.
import os, subprocess
if not globals().get("ACCESS_OK"):
    raise SystemExit("STOP: cell 3 did not pass. Nothing to download.")
if not PROJECT:
    raise SystemExit("STOP: make_val_subset.py needs PROJECT set (the data bucket is requester-pays).")
VAL_DEST = "/content/ttt-data/llama3-books3"
env = dict(os.environ, GCP_BILLING_PROJECT=PROJECT)
base = ["python3", "/content/TTT/scripts/make_val_subset.py", "--dest", VAL_DEST]
run(base + ["fetch", "--tokens", str(VAL_TOKENS), "--probe-only"], env=env)
run(base + ["fetch", "--tokens", str(VAL_TOKENS)], env=env)
run(base + ["covers", "--tokens", str(VAL_TOKENS)], env=env)

# /train chunk 0: the attacker's span corpus for C1b (prepare_phase1.sh -> dump_tokens.py).
# Same function the box would have called, kept out of llama3-books3 so the 000 store is untouched.
import sys
sys.path.insert(0, "/content/TTT/scripts")
import dump_tokens
train_dir = dump_tokens.fetch_train_chunk0(PROJECT, "gs://llama3-books3", __import__("pathlib").Path("/content/ttt-data/train-zarr"))
print("train chunk 0 at", train_dir)

run(["tar", "-cf", "/content/drive/MyDrive/ttt/ttt-books3-val.tar", "-C", "/content/ttt-data", "llama3-books3", "train-zarr"])
drive.flush_and_unmount()
print("DONE: Drive -> MyDrive/ttt/ttt-books3-val.tar (val subset + train chunk 0)")

When cell 5 prints **DONE**: Runtime -> Disconnect and delete runtime. Share the two tars and the
manifest from `MyDrive/ttt/` with the person running the session.

On the machine that runs the session (for 125M, the laptop under WSL2; runbook Part B-local), extract
both tars and point the bootstrap at them:

```
tar -xf ttt-handoff-<CKPT>.tar -C ~/ttt-data
tar -xf ttt-books3-val.tar     -C ~/ttt-data        # gives ~/ttt-data/llama3-books3 and ~/ttt-data/train-zarr
CKPT_DIR=~/ttt-data/ttt-handoff/<CKPT> CKPT_SHA_MANIFEST=~/ttt-data/ttt-handoff/checkpoint-sha256-<CKPT>.txt DATA_ROOT=~/ttt-data LOCAL=1 bash scripts/bootstrap_gpu_box.sh
```

Later, for C1b: `TRAIN_DIR=~/ttt-data/train-zarr bash scripts/prepare_phase1.sh` (no billing project needed).

The bootstrap should then report `gcs : not needed -- checkpoint and val subset both on local disk`.